In [1]:
from collections import defaultdict, deque
import math
import random
from DualGraph import DualGraph
from Tile import Tile
from Configuration import Configuration
from MinCostSolver import MinCostFlow
from utils import *

In [2]:
def compute_initial_mosaic_drawing(GM, tiling):
    """
    Section 4.1
    Computes a simple mosaic drawing DT(GM).

    GM     : planar triangulated dual graph
    tiling : square or hex grid abstraction

    Returns:
        dict[vertex] -> Configuration
    """
    # Placeholder for Schnyder-based orderly spanning tree
    spanning_tree = compute_schnyder_spanning_tree(GM)

    configurations = {}
    for v in GM.vertices:
        configurations[v] = Configuration(v)

    # Stretch to visibility drawing, then fill gaps (CLL05)
    # Implementation detail omitted (graph drawing literature)
    assign_tiles_from_visibility_drawing(configurations, spanning_tree, tiling)

    return configurations


In [3]:
def compute_guiding_shape(face_polygon, target_tiles, tiling):
    """
    Section 4.2 – Guiding shapes

    face_polygon : geometry of original region F(v)
    target_tiles : n_S(v)
    tiling       : grid abstraction

    Returns:
        set[Tile]
    """
    # Scale face to desired area
    scaled_face = scale_polygon_to_area(
        face_polygon,
        target_tiles * tiling.tile_area
    )

    best_shape = None
    best_score = float("inf")

    for offset in generate_small_translations():
        shifted = translate_polygon(scaled_face, offset)
        candidate = select_tiles_with_max_overlap(shifted, tiling, target_tiles)
        score = symmetric_difference_area(candidate, shifted)

        if score < best_score:
            best_shape = candidate
            best_score = score

    return best_shape


In [4]:
def move_and_reshape(configurations, guiding_shapes, GM, max_iters=500):
    """
    Section 4.2 – MOVEANDRESHAPE
    """
    for _ in range(max_iters):
        changed = False

        # --- Reshaping step ---
        for v in GM.vertices:
            C = configurations[v]
            S = guiding_shapes[v]

            for tile in list(C.tiles):
                for u in GM.neighbors(v):
                    if try_set_tile(tile, v, u, configurations, guiding_shapes):
                        changed = True

        # --- Moving step ---
        if not changed:
            moved = force_directed_move(guiding_shapes, GM)
            if not moved:
                break


In [5]:
def try_set_tile(tile, from_v, to_v, configurations, guiding_shapes):
    """
    Executes set(t, v) if valid and improves objective.
    """
    C_from = configurations[from_v]
    C_to   = configurations[to_v]

    if tile not in C_from.tiles:
        return False

    # Temporarily move tile
    C_from.tiles.remove(tile)
    C_to.tiles.add(tile)

    valid = (
        C_from.is_connected() and
        C_to.is_connected() and
        adjacency_preserved(configurations)
    )

    if not valid:
        # rollback
        C_to.tiles.remove(tile)
        C_from.tiles.add(tile)
        return False

    # Check normalized symmetric difference improvement
    if improves_max_delta(from_v, to_v, configurations, guiding_shapes):
        return True

    # rollback
    C_to.tiles.remove(tile)
    C_from.tiles.add(tile)
    return False


In [6]:
def force_directed_move(guiding_shapes, GM, step=0.01):
    """
    Section 4.2 – ForceDirectedAlgorithm
    """
    forces = defaultdict(lambda: [0.0, 0.0])

    for v in GM.vertices:
        for u in GM.neighbors(v):
            dist = tile_distance(guiding_shapes[v], guiding_shapes[u])
            fx, fy = attraction_force(guiding_shapes[v], guiding_shapes[u], dist)
            forces[v][0] += fx
            forces[v][1] += fy

        for u in GM.vertices:
            if u == v:
                continue
            overlap = guiding_shapes[v] & guiding_shapes[u]
            if overlap:
                fx, fy = repulsion_force(v, u, overlap, guiding_shapes)
                forces[v][0] += fx
                forces[v][1] += fy

    moved = False
    for v, (fx, fy) in forces.items():
        if fx != 0 or fy != 0:
            translate_guiding_shape(guiding_shapes[v], fx * step, fy * step)
            moved = True

    return moved


In [7]:
def correct_configuration_sizes(configurations, guiding_shapes, tiling):
    """
    Section 4.3 – Minimum Cost Flow correction
    """
    flow_network = build_flow_network(configurations, guiding_shapes, tiling)
    flow = solve_min_cost_flow(flow_network)

    apply_flow_to_configurations(flow, configurations)


In [8]:
def build_flow_network(configurations, guiding_shapes, tiling):
    """
    Section 4.3 – Flow network D = (N, A)
    """
    network = MinCostFlow()

    # Boundary nodes
    for v, C in configurations.items():
        for tile in boundary_tiles(C, tiling):
            network.add_node(tile, capacity=1)

    # Supply nodes
    for v, C in configurations.items():
        supply = len(C.tiles) - len(guiding_shapes[v])
        network.add_supply_node(v, supply)

    network.add_supply_node("sea", total_negative_supply(configurations))

    # Adjacency arcs
    for s, t in adjacent_boundary_pairs(configurations):
        cost = compute_transfer_cost(s, t, guiding_shapes)
        network.add_edge(s, t, capacity=1, cost=cost)

    return network


In [9]:
def mosaic_cartogram(map_data, weights, tiling):
    """
    Complete algorithm (Section 4)
    """
    GM = build_dual_graph(map_data, weights)

    configurations = compute_initial_mosaic_drawing(GM, tiling)

    guiding_shapes = {
        # v: compute_guiding_shape(
        #     map_data.face(v),
        #     weights[v],
        #     tiling
        # )
        # for v in GM.vertices
        v: set(compute_guiding_shape(
            random.sample(
            map_data.face(v)),
            weights[v]))
        for v in GM.vertices
    }

    move_and_reshape(configurations, guiding_shapes, GM)

    correct_configuration_sizes(configurations, guiding_shapes, tiling)

    return configurations


In [10]:
def build_grid(width, height):
    tiles = {}
    for x in range(width):
        for y in range(height):
            tiles[(x, y)] = Tile(x, y)

    for (x, y), t in tiles.items():
        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
            if (x+dx, y+dy) in tiles:
                t.neighbors.add(tiles[(x+dx, y+dy)])
    return tiles


In [11]:
# Build grid
tiling = build_grid(6, 6)

# Define regions
weights = {
    "A": 3,
    "B": 4,
    "C": 2
}

# Adjacency (triangle)
map_data = [
    ("A", "B"),
    ("B", "C"),
    ("C", "A")
]

configs = mosaic_cartogram(map_data, weights, tiling)

for v, C in configs.items():
    print(v, C.tiles)


AttributeError: 'list' object has no attribute 'face'